---
jupyter: python3
---


# Exercise 1: List patients using the get method from requests_fhir

The file request_fhir.py should be in the same folder as this markdown file.
From the exercise:

In [ ]:
import json
import request_fhir as requests
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = 'https://test/fhir'
json_headers = {'content-type': 'application/json', 'accept': 'text/plain'}

Check if it works, the output should be related to error 501 (didn't actually ask for any information)

In [ ]:
print(requests.get(BASE_URL))

Retrieving the full patient list:

In [ ]:
patients_url = f'{BASE_URL}/Patient'

print(patients_url)

response = requests.get(patients_url)

bundle = json.loads(response._content)
if bundle['resourceType'] == 'Bundle' and bundle['type'] == 'searchset':
    print(f"Found {len(bundle.get('entry', []))} patients:")
    res = pd.DataFrame(columns=['id', 'sex', 'd.birth'])

    for entry in bundle.get('entry', []):
        patient = entry.get('resource')
        if patient and patient['resourceType'] == 'Patient':
            patient_id = patient.get('id', 'N/A')
            sex = patient.get('gender', 'N/A')
            birthdate = patient.get('birthDate', 'N/A')
            
            res = pd.concat([res, pd.DataFrame([{
                'id': patient_id,
                'sex': sex,
                'd.birth': birthdate
            }])], ignore_index=True)

res

# Exercise 2: Look at the snomed.csv file inside the data folder to see which code to use to access the data

In [ ]:
# Read the SNOMED CSV
snomed = pd.read_csv("data/snomed.csv")

# Use the dictionary data to translate the 
dict_data = pd.read_csv("Data/dict_data.csv")

# Quick translation using diag data set
snomed.loc[29:31, 'label'] = dict_data['diag'].values

# Print the possible predictors and their corresponding codes
snomed[['code','label']]

# Exercise 3: Get measurements of cholesterol for a patient and plot it over time

In [ ]:
# Get cholesterol code
chol_code = snomed[snomed['label'].str.contains('chol')]['code']

# Select an id (we chose 6)
patient_id = 6

# Combine it in an URL
chol_url = f'{BASE_URL}/Observation?patient={patient_id}&code={chol_code.iloc[0]}'

# Get the information and read it 
response = requests.get(chol_url)

cholesterol_bundle = json.loads(response._content)
entry = cholesterol_bundle.get('entry')

# Iterate over the entries (each cholesterol observation for person 6)
for entry in cholesterol_bundle.get('entry', []):
  resource = entry.get('resource')
  if resource and resource['resourceType'] == 'Observation':
    date = resource.get('effectiveDateTime')
    value = resource.get('valueQuantity', {}).get('value')
    if date and value:
      observations.append({'date': pd.to_datetime(date), 'value': float(value)})

# Combine the above in a DataFrame
observations_df = pd.DataFrame(observations)
observations_df = observations_df.sort_values(by='date')
       
# Plot the observations
plt.figure(figsize=(10, 6))
plt.plot(observations_df['date'], observations_df['value'], marker='o', linestyle='-')
plt.xlabel('Date')
plt.ylabel('Cholesterol Value')
plt.title(f'Cholesterol Levels Over Time for Patient {patient_id}')
plt.grid()
plt.show()

# Exercise 4: Obtain the necessary information for your model using requests_fhir (chatgpt'ed, not tested)
For our model, we need all possible information in the snomed file.

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
from dateutil import parser
from datetime import datetime
from dateutil.relativedelta import relativedelta
import random

# Utilities
def get_r(url):
    response = requests.get(url)
    return response.json()

def func_nyears(series, func, dates, n_years):
    # Apply function over the past n years for each timepoint
    result = []
    for i in range(len(series)):
        current_time = parser.parse(dates[i])
        mask = [(parser.parse(dates[j]) >= current_time - relativedelta(years=n_years)) and 
                (parser.parse(dates[j]) <= current_time) 
                for j in range(len(series))]
        values = np.array(series)[mask]
        result.append(func(values))
    return result

# Sample data preparation (just the simulation part)
random.seed(2025)
ids = random.sample(list(patient_df["id"]), len(patient_df["id"]) // 4)
ids_str = ",".join(map(str, ids))

# Visit data
vst_code = snomed[snomed["source"] == "visit_date"]["code"].values[0]
vst_url = f"{base}/Observation?patient={ids_str}&code={vst_code}"
vst_data = get_r(vst_url)

vst_df = pd.json_normalize(vst_data['entry'])
vst_df = vst_df[["resource.subject.reference", "resource.effectiveDateTime", "resource.valueQuantity.value"]]
vst_df.columns = ["id", "date", "visit"]
vst_df["id"] = vst_df["id"].str.replace("Patient/", "").astype(int)

# Baseline data
bsl_code = snomed[snomed["source"] == "baseline_data"]["code"].values[0]
bsl_url = f"{base}/Observation?patient={ids_str}&code={bsl_code}"
bsl_data = get_r(bsl_url)

bsl_df = pd.json_normalize(bsl_data['entry'])
bsl_df = bsl_df[["resource.subject.reference", "resource.effectiveDateTime", "resource.valueQuantity.value"]]
bsl_df.columns = ["id", "date", "smoking"]
bsl_df["id"] = bsl_df["id"].str.replace("Patient/", "").astype(int)

# Other observations
obs_codes = snomed[snomed["source"].isin(["blood_data", "quest_data", "diag_data"])]["code"]
obs_df = pd.DataFrame()

for code in obs_codes:
    tmp_url = f"{base}/Observation?patient={ids_str}&code={code}"
    tmp_data = get_r(tmp_url)
    tmp_df = pd.json_normalize(tmp_data['entry'])[
        ["resource.subject.reference", "resource.effectiveDateTime", "resource.valueQuantity.value"]
    ]
    tmp_df.columns = ["id", "date", "value"]
    tmp_df["id"] = tmp_df["id"].str.replace("Patient/", "").astype(int)
    tmp_df["value"] = pd.to_numeric(tmp_df["value"], errors="coerce")
    var_name = snomed[snomed["code"] == code]["label"].values[0]
    tmp_df.rename(columns={"value": var_name}, inplace=True)

    if obs_df.empty:
        obs_df = tmp_df
    else:
        obs_df = pd.merge(obs_df, tmp_df, on=["id", "date"], how="outer")

# Procedure data
pro_codes = snomed[snomed["source"].isin(["treat_data", "events"])]["code"]
pro_df = pd.DataFrame()

for code in pro_codes:
    tmp_url = f"{base}/Procedure?patient={ids_str}&code={code}"
    tmp_data = get_r(tmp_url)
    tmp_df = pd.json_normalize(tmp_data['entry'])[
        ["resource.subject.reference", "resource.performedDateTime", "resource.valueQuantity.value"]
    ]
    tmp_df.columns = ["id", "date", "value"]
    tmp_df["id"] = tmp_df["id"].str.replace("Patient/", "").astype(int)
    tmp_df["value"] = pd.to_numeric(tmp_df["value"], errors="coerce")
    var_name = snomed[snomed["code"] == code]["label"].values[0]
    tmp_df.rename(columns={"value": var_name}, inplace=True)

    if pro_df.empty:
        pro_df = tmp_df
    else:
        pro_df = pd.merge(pro_df, tmp_df, on=["id", "date"], how="outer")

# Combine all
newdata = pro_df.copy()
newdata = newdata.sort_values(["id", "date"])

# Fill and merge
newdata = newdata.fillna(method="ffill")
for col in ["aspirin", "statins"]:
    if col in newdata.columns:
        newdata[col].fillna(newdata[col].mean(), inplace=True)

newdata = newdata.merge(obs_df, on=["id", "date"], how="outer")
newdata = newdata.merge(bsl_df, on=["id", "date"], how="outer")
newdata = newdata.merge(vst_df, on=["id", "date"], how="outer")
newdata = newdata.merge(patient_df, on="id", how="left")
newdata["date"] = pd.to_datetime(newdata["date"])

# Final transformation
new_final_data = newdata.sort_values(["id", "date"])
new_final_data["visit"] = new_final_data["visit"].notna()
new_final_data["age"] = (new_final_data["date"] - pd.to_datetime(new_final_data["d.birth"])).dt.total_seconds() / (60*60*24*365.25)
new_final_data["male"] = (new_final_data["sex"] == "male").astype(int)
new_final_data["smoker_current"] = (new_final_data["smoking"] == "current").astype(int)
new_final_data["smoker_former"] = (new_final_data["smoking"] == "former").astype(int)

# One-hot encoding and cumulative sums
for col in ["diabetes", "hyperlipidemia", "hypertension"]:
    new_final_data[col] = new_final_data[col].fillna(0).astype(int)
    new_final_data[col] = new_final_data.groupby("id")[col].cumsum()

for event in ["revascularization", "malignancy", "stroke", "amputation", "infarction"]:
    new_final_data[event] = new_final_data[event].fillna(0).astype(int)
    new_final_data[f"n_{event}"] = new_final_data.groupby("id")[event].cumsum()
    new_final_data[f"nyear_{event}"] = new_final_data.groupby("id").apply(
        lambda group: func_nyears(group[event].tolist(), sum, group["date"].astype(str).tolist(), n_years)
    ).explode().astype(float).reset_index(level=0, drop=True)

# Final processing continues...

# Normalize (assuming fin_sum has min and max for each variable)
for col in new_final_data.columns.difference(["id", "date"]):
    min_col = f"min_{col}"
    max_col = f"max_{col}"
    if min_col in fin_sum.columns and max_col in fin_sum.columns:
        new_final_data[col] = (new_final_data[col] - fin_sum[min_col]) / (fin_sum[max_col] - fin_sum[min_col])

# Exercise 5: Feed these data into your trained model and make predictions

In [ ]:
# Load model
loaded_model = joblib.load("trained_model.joblib")